# Day 14 — Union-Find (Disjoint Set Union)

Some questions are not about *paths* at all. "Are these two accounts the same
person?", "how many separate networks are left after this cable fails?", "does
adding this edge close a cycle?" — all you need is a way to keep a pile of
elements partitioned into groups, merge two groups, and ask whether two
elements are in the same group.

**Union-Find** does exactly that, in a plain integer array, in effectively
constant time per operation.


## The forest, and the array that is the forest

Every set is a tree. Each element stores one number: its parent. A root points
at itself, and that root is the **id** of the set — so `find(x)` walks up to
the root and `x` and `y` are in the same set exactly when their roots match.

The picture and the array below are the same object.


In [1]:
parent = [0, 0, 0, 2, 4, 4, 6]

def find_plain(x):
    while parent[x] != x:
        x = parent[x]
    return x

for i in range(len(parent)):
    print(f'  {i} -> parent {parent[i]}   root {find_plain(i)}')


  0 -> parent 0   root 0
  1 -> parent 0   root 0
  2 -> parent 0   root 0
  3 -> parent 2   root 0
  4 -> parent 4   root 4
  5 -> parent 4   root 4
  6 -> parent 6   root 6


## Two lines of defence

A naive `union` that always hangs `y`'s root under `x`'s root is correct, but a
few unlucky merges build one long chain and every `find` becomes O(n). Two
tricks fix it, and they are both about three lines long.

**Union by rank (or size)** — always attach the shorter tree under the taller
one, so the height only grows when two equally tall trees merge.

**Path compression** — after `find(x)` has located the root, walk the path a
second time and point every node on it straight at the root. The work you just
did is never repeated.

Together they give an amortised cost of O(α(n)) per operation, where α is the
inverse Ackermann function. α(n) ≤ 4 for any n that fits in this universe, so
in practice it is constant.


In [2]:
class UnionFind:
    """Disjoint set union over 0..n-1."""

    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n          # upper bound on the tree height
        self.size = [1] * n          # elements in the set (roots only)
        self.count = n               # number of disjoint sets

    def find(self, x):
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:            # path compression
            self.parent[x], x = root, self.parent[x]
        return root

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return False                          # already together
        if self.rank[rx] < self.rank[ry]:         # union by rank
            rx, ry = ry, rx
        self.parent[ry] = rx
        self.size[rx] += self.size[ry]
        if self.rank[rx] == self.rank[ry]:        # only a tie grows the height
            self.rank[rx] += 1
        self.count -= 1
        return True

    def connected(self, x, y):
        return self.find(x) == self.find(y)

    def set_size(self, x):
        return self.size[self.find(x)]

uf = UnionFind(7)
for a, b in [(0, 1), (0, 2), (2, 3), (4, 5)]:
    uf.union(a, b)
    print(f'  union({a}, {b})  ->  parent = {uf.parent}   sets = {uf.count}')

print()
print('  connected(3, 1) =', uf.connected(3, 1))
print('  connected(3, 5) =', uf.connected(3, 5))
print('  set_size(3)     =', uf.set_size(3))


  union(0, 1)  ->  parent = [0, 0, 2, 3, 4, 5, 6]   sets = 6
  union(0, 2)  ->  parent = [0, 0, 0, 3, 4, 5, 6]   sets = 5
  union(2, 3)  ->  parent = [0, 0, 0, 0, 4, 5, 6]   sets = 4
  union(4, 5)  ->  parent = [0, 0, 0, 0, 4, 4, 6]   sets = 3

  connected(3, 1) = True
  connected(3, 5) = False
  set_size(3)     = 4


Note what `union` returns. `False` means the two elements were **already** in
the same set — which is precisely "this edge closes a cycle". Kruskal's minimum
spanning tree algorithm is little more than sorting the edges and keeping the
ones where `union` returns `True`.


## Watching path compression flatten a chain

Build the worst case by hand — `3 -> 2 -> 1 -> 0` — and call `find(3)` once.


In [3]:
chain = UnionFind(4)
chain.parent = [0, 0, 1, 2]

print('  before find(3):', chain.parent)
chain.find(3)
print('  after  find(3):', chain.parent, '  <- everything points at the root')


  before find(3): [0, 0, 1, 2]
  after  find(3): [0, 0, 0, 0]   <- everything points at the root


## How much does it actually save?

Chain 20,000 elements together in the worst possible order, then measure how
many pointer hops a full sweep of `find` costs.


In [4]:
class NaiveUnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.steps = 0

    def find(self, x):
        while self.parent[x] != x:
            self.steps += 1
            x = self.parent[x]
        return x

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx != ry:
            self.parent[ry] = rx        # blind attach - builds a chain

n = 20000

naive = NaiveUnionFind(n)
for i in range(1, n):
    naive.union(i, i - 1)
naive.steps = 0
for i in range(n):
    naive.find(i)

smart = UnionFind(n)
for i in range(1, n):
    smart.union(i, i - 1)
hops = 0
for i in range(n):
    x = i
    while smart.parent[x] != x:
        hops += 1
        x = smart.parent[x]

print(f'  naive   : {naive.steps:>12,} pointer hops for {n:,} finds')
print(f'  rank+pc : {hops:>12,} pointer hops for {n:,} finds')
print(f'  ratio   : {naive.steps / max(hops, 1):>12,.0f}x')


  naive   :  199,990,000 pointer hops for 20,000 finds
  rank+pc :       19,999 pointer hops for 20,000 finds
  ratio   :       10,000x


## LeetCode 547 — Number of Provinces

`isConnected[i][j] == 1` means city *i* and city *j* are directly linked; a
province is a connected component. Union every linked pair and read off the
number of sets left.

The matrix is symmetric with a diagonal of ones, so only the strict upper
triangle carries information — scanning all n² entries is not wrong, just
twice the work.


In [5]:
def find_circle_num(is_connected):
    n = len(is_connected)
    uf = UnionFind(n)
    for i in range(n):
        for j in range(i + 1, n):        # upper triangle only
            if is_connected[i][j]:
                uf.union(i, j)
    return uf.count

cases = [
    ([[1, 1, 0], [1, 1, 0], [0, 0, 1]], 2),
    ([[1, 0, 0], [0, 1, 0], [0, 0, 1]], 3),
    ([[1, 1, 0, 0, 0],
      [1, 1, 0, 0, 0],
      [0, 0, 1, 1, 0],
      [0, 0, 1, 1, 0],
      [0, 0, 0, 0, 1]], 3),
]

for grid, want in cases:
    for row in grid:
        print('    ' + ' '.join(str(v) for v in row))
    got = find_circle_num(grid)
    print(f'    -> {got} province(s)   {"ok" if got == want else "WRONG"}')
    print()
    assert got == want


    1 1 0
    1 1 0
    0 0 1
    -> 2 province(s)   ok

    1 0 0
    0 1 0
    0 0 1
    -> 3 province(s)   ok

    1 1 0 0 0
    1 1 0 0 0
    0 0 1 1 0
    0 0 1 1 0
    0 0 0 0 1
    -> 3 province(s)   ok



BFS/DFS would answer this one too, in the same O(n²) — the matrix has to be
read either way. Union-Find earns its keep when the edges **arrive over time**:
a traversal has to re-run from scratch after every new edge, while Union-Find
just absorbs it.


## Complexity

| Operation | Naive | Rank + path compression |
|---|---|---|
| `find` | O(n) worst case | O(α(n)) amortised |
| `union` | O(n) worst case | O(α(n)) amortised |
| space | O(n) | O(n) |

α(n) is the inverse Ackermann function: it is at most 4 for any input size that
will ever exist, so every operation is constant time for practical purposes.


## Test it

A randomised cross-check against an obviously-correct model: keep the real
partition as a list of Python sets and assert that every `connected()` query
agrees.


In [6]:
from random import Random

rng = Random(14)
for trial in range(200):
    n = rng.randint(1, 30)
    uf = UnionFind(n)
    model = [{i} for i in range(n)]
    for _ in range(n * 2):
        a, b = rng.randrange(n), rng.randrange(n)
        uf.union(a, b)
        sa = next(s for s in model if a in s)
        sb = next(s for s in model if b in s)
        if sa is not sb:
            sa |= sb
            model.remove(sb)
        assert uf.count == len(model)
        for x in range(n):
            for y in range(n):
                same = any(x in s and y in s for s in model)
                assert uf.connected(x, y) == same

print('  200 random trials, every connected() query matched the model')
print('  all assertions passed')


  200 random trials, every connected() query matched the model
  all assertions passed


## Where this comes back

Union-Find is the engine inside Kruskal's minimum spanning tree, and it shows
up wherever "same group?" is the whole question: percolation and image
segmentation, Hoshen–Kopelman connected-component labelling, type inference
unifying type variables, compilers merging equivalence classes of expressions,
and offline dynamic connectivity. Whenever you catch yourself re-running a
traversal after every new edge, this is the structure you wanted.
